In [ ]:
# Using PyTorch 1.4

import numpy as np
import pyarrow.parquet as pq
import pandas as pd
import random
import torch
import fire
import logging
import os
import csv
import re
import shutil

from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer, GPT2LMHeadModel, AdamW, get_linear_schedule_with_warmup
from torch.optim import SGD
from tqdm import tqdm, trange
import torch.nn.functional as F
from private_transformers import PrivacyEngine
from torch.nn.utils.rnn import pad_sequence


In [ ]:

YAML_boolq_string = """
group: multimedqa
task:
  - pubmedqa
  - medmcqa
  - medqa_4options
  - task: mmlu_anatomy
    task_alias: "anatomy (mmlu)"
  - task: mmlu_clinical_knowledge
    task_alias: "clinical_knowledge (mmlu)"
  - task: mmlu_college_medicine
    task_alias: "college_medicine (mmlu)"
  - task: mmlu_medical_genetics
    task_alias: "medical_genetics (mmlu)"
  - task: mmlu_professional_medicine
    task_alias: "professional_medicine (mmlu)"
  - task: mmlu_college_biology
    task_alias: "college_biology (mmlu)"
aggregate_metric_list:
  - metric: acc
    aggregation: mean
    weight_by_size: True
"""
with open("multimedqa.yaml", "w") as f:
    f.write(YAML_boolq_string)

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss(reduction="none") 

def loss_function(outputs, input_tensor):
    logits = outputs.logits  # (batch_size, seq_len, vocab_size)
    shift_logits = logits[:, :-1, :].contiguous()  # Alinear la secuencia
    shift_labels = input_tensor[:, 1:].contiguous() 

    loss_per_example = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)), 
        shift_labels.view(-1),  # (batch_size * seq_len)
        reduction="none" 
    )

    return loss_per_example.view(input_tensor.size(0), -1).mean(dim=1)  


In [ ]:
import torch
import csv
import os
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.optim import SGD
from transformers import GPT2Tokenizer, get_linear_schedule_with_warmup
device = "cuda"


class CSVMIMIC(Dataset):
    def __init__(self, control_code, truncate=False, gpt2_type="gpt2", max_length=768):
        self.tokenizer = GPT2Tokenizer.from_pretrained(gpt2_type)
        self.tokenizer.pad_token = self.tokenizer.eos_token  # Usar EOS como padding
        self.notes = []


        with open("MIMIC_NoteEvents_deanonymized.csv", newline="", encoding="ISO-8859-1") as csvfile:
            tweet_csv = csv.reader(csvfile)
            for row in tweet_csv:
                encoded = self.tokenizer.encode(f"<|{control_code}|>{row[10][:max_length]}<|endoftext|>")
                self.notes.append(torch.tensor(encoded, dtype=torch.long))  # Convertir a tensor long

        if truncate:
            self.notes = self.notes[:20000]

    def __len__(self):
        return len(self.notes)

    def __getitem__(self, idx):
        return self.notes[idx]

# Función para aplicar padding en DataLoader
def collate_fn(batch):
    batch_padded = pad_sequence(batch, batch_first=True, padding_value=tokenizer.pad_token_id)
    return batch_padded.to(device)


def train(
    dataset,
    model,
    batch_size=4,
    epochs=4,
    lr=2e-5,
    warmup_steps=5000,
    device="cuda",
    output_dir=".",
    output_prefix="wreckgar",
    save_model_on_epoch=False,
):
    model = model.to(device)
    model.train()

    optimizer = SGD(model.parameters(), lr=lr)

    privacy_engine = PrivacyEngine(
        model,
        batch_size=batch_size,
        sample_size=len(dataset),
        epochs=epochs,
        max_grad_norm=0.1,
        target_epsilon=3,
        clipping_mode="ghost",
    )
    privacy_engine.attach(optimizer)

    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=-1
    )

    # Crear DataLoader con la función de padding
    train_dataloader = DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        collate_fn=collate_fn
    )

    for epoch in range(epochs):
        print(f"Training epoch {epoch}")

        for idx, input_tensors in tqdm(enumerate(train_dataloader)):
            input_tensors = input_tensors.to(device)


            outputs = model(input_tensors, labels=input_tensors)
            loss = outputs.loss

            optimizer.zero_grad()
            optimizer.step()
            scheduler.step()

        if save_model_on_epoch:
            torch.save(
                model.state_dict(),
                os.path.join(output_dir, f"{output_prefix}-{epoch}.pt"),
            )

    return model


In [ ]:
dataset = CSVMIMIC("<|note|>", truncate=True, gpt2_type="gpt2")

In [ ]:
gpt2_type = "gpt2"

In [ ]:
YAML_boolq_string = """
group: multimedqa
task:
  - pubmedqa
aggregate_metric_list:
  - metric: acc
    aggregation: mean
    weight_by_size: True
"""
with open("multimedqa.yaml", "w") as f:
    f.write(YAML_boolq_string)

In [ ]:

learning_rates = [3e-5, 3e-4, 3e-2, 3e-1] 
batch_sizes= [8,16,32, 46] 
warmup_steps= [2500, 5000, 8000] 


model = GPT2LMHeadModel.from_pretrained("gpt2")

for warmupIndex in range(len(warmup_steps)):
    for batchIndex in range(len(batch_sizes)):
        for lrIndex in range(len(learning_rates)):
            trained_model = train(dataset=dataset, model=model)
            
            modelDir = 'model'
            trained_model.save_pretrained(modelDir)
            tokenizer=GPT2Tokenizer.from_pretrained(gpt2_type)
            tokenizer.save_pretrained(modelDir)
            result = !lm_eval \
            --model hf \
            --model_args pretrained=model \
            --output_path /Output \
            --include_path ./ \
            --tasks multimedqa \
            --device cpu \
            --trust_remote_code



            log_text = ' '.join(result)
            match = re.search(r'pubmedqa\s+\|[\s\S]+?\|([\d.]+)', log_text[250900:])

            if match:
                accuracy = float(match.group(1))
                print(f"Extracted Accuracy: {accuracy}")
            else:
                accuracy = print("Accuracy not found in log.")

            shutil.rmtree('model')
            with open("results", "a") as f:
                f.write(f'{{ "learningRate": {learning_rates[lrIndex]}, "batchSize": {batch_sizes[batchIndex]}, "warmupSteps": {warmup_steps[warmupIndex]}, "accuracy": {accuracy} }}\n')

